In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

start_time = spark.sql(
    "SELECT current_timestamp()"
).first()[0]
crypto_schema = StructType([
    StructField("high", DoubleType(), True),
    StructField("open", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("timestamp", LongType(), True),
    StructField("ticker", StringType(), True)
])

print("Schema Created Successfully")
# Created the schema

# Read csv file and define dataframe
df = spark.read \
    .schema(crypto_schema) \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/crypto_data/")
df.count()

# Add the timestamp column
bronze_df = df.withColumn(
    "load_timestamp",
    current_timestamp()
)

bronze_df.count()

# Store processed Bronze data as a Delta table
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_crypto_market")

# Check that data was loaded successfully into the Bronze table
spark.sql("""
SELECT *
FROM bronze_crypto_market
LIMIT 5
""").show()

silver_df = bronze_df.withColumn(
    "observation_date",
    to_date(from_unixtime(col("timestamp")))
)
silver_df.select(
    "ticker",
    "timestamp",
    "observation_date"
).show(10, False)

Schema Created Successfully
+------------------+------------------+------------------+------------------+--------+----------+-------------+--------------------+
|              high|              open|            volume|             close|     low| timestamp|       ticker|      load_timestamp|
+------------------+------------------+------------------+------------------+--------+----------+-------------+--------------------+
|0.9995390176773072| 1.000100016593933|0.9994590282440186| 1.000581979751587| 75238.0|1738454400|XUSD20317-USD|2026-10-10 19:46:...|
|1.0002609491348269|0.9995390176773072|0.9986220002174376| 1.000802993774414|488773.0|1738540800|XUSD20317-USD|2026-10-10 19:46:...|
|0.9999099969863892|1.0002609491348269|0.9987059831619264| 1.000301957130432|123801.0|1738627200|XUSD20317-USD|2026-10-10 19:46:...|
|0.9998030066490172|0.9999099969863892| 0.999684989452362|1.0002189874649048| 68226.0|1738713600|XUSD20317-USD|2026-10-10 19:46:...|
|0.9999250173568726|0.9998030066490172| 0

In [0]:
from pyspark.sql.functions import *

# Split ticker into coin symbol and quote currency for easier analysis

silver_df = silver_df \
    .withColumn(
        "coin_symbol",
        split(col("ticker"), "-")[0]
    ) \
    .withColumn(
        "quote_currency",
        split(col("ticker"), "-")[1]
    )

silver_df.select(
    "ticker",
    "coin_symbol",
    "quote_currency"
).show(10, False)


+------------+-----------+--------------+
|ticker      |coin_symbol|quote_currency|
+------------+-----------+--------------+
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
+------------+-----------+--------------+
only showing top 10 rows


In [0]:
# Select required columns and rename price fields for a clean Silver layer schema
silver_df = silver_df.select(
    "coin_symbol",
    "quote_currency",
    "observation_date",
    col("open").alias("open_price"),
    col("high").alias("high_price"),
    col("low").alias("low_price"),
    col("close").alias("close_price"),
    "volume",
    "load_timestamp"
).withColumn(
    "market_cap",
    lit(None).cast("double")
)
silver_df.printSchema()

root
 |-- coin_symbol: string (nullable = true)
 |-- quote_currency: string (nullable = true)
 |-- observation_date: date (nullable = true)
 |-- open_price: double (nullable = true)
 |-- high_price: double (nullable = true)
 |-- low_price: double (nullable = true)
 |-- close_price: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)
 |-- market_cap: double (nullable = true)



In [0]:
# Remove duplicate records based on coin and observation date
silver_df = silver_df.dropDuplicates(
    ["coin_symbol", "observation_date"]
)
silver_df.count()

8114768

In [0]:
silver_df.write \
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("silver_crypto_market")

spark.sql("""
SELECT *
FROM silver_crypto_market
LIMIT 5
""").show()




+-----------+--------------+----------------+-----------------+----------+-----------------+-----------------+------------------+--------------------+----------+
|coin_symbol|quote_currency|observation_date|       open_price|high_price|        low_price|      close_price|            volume|      load_timestamp|market_cap|
+-----------+--------------+----------------+-----------------+----------+-----------------+-----------------+------------------+--------------------+----------+
|   SAFEBULL|           USD|      2022-05-11|              0.0|    3583.0|              0.0|              0.0|               0.0|2026-10-10 19:46:...|      NULL|
|   SAFEBULL|           USD|      2025-05-11|              0.0|       0.0|              0.0|              0.0|               0.0|2026-10-10 19:46:...|      NULL|
|  SATO36386|           USD|      2026-04-29|              0.0|       0.0|              0.0|              0.0|               0.0|2026-10-10 19:46:...|      NULL|
|        SDM|           USD|

In [0]:
# Read CoinGecko incremental JSON file
incremental_file = "markets(2).json"
incremental_df = spark.read.json(
    f"/Volumes/workspace/default/crypto_data/{incremental_file}"
)

# Display sample records
incremental_df.show(5)

# Display schema
incremental_df.printSchema()

# Count records
incremental_df.count()

+--------+---------------------+--------------------+----------+---------------------+--------------------+--------------------+-------------+-----------------------+--------+-----------+--------------------+--------------------+--------+-------------+---------------------+--------------------------------+---------------+----------+--------+-------------------+---------------------------+--------------------+------+--------------------+---------------+
|     ath|ath_change_percentage|            ath_date|       atl|atl_change_percentage|            atl_date|  circulating_supply|current_price|fully_diluted_valuation|high_24h|         id|               image|        last_updated| low_24h|   market_cap|market_cap_change_24h|market_cap_change_percentage_24h|market_cap_rank|max_supply|    name|   price_change_24h|price_change_percentage_24h|                 roi|symbol|        total_supply|   total_volume|
+--------+---------------------+--------------------+----------+---------------------+

250

In [0]:
from pyspark.sql.functions import *
incremental_silver_df = incremental_df.select(
    upper(col("symbol")).alias("coin_symbol"),

    lit("USD").alias("quote_currency"),

    to_date(col("last_updated")).alias("observation_date"),

    lit(None).cast("double").alias("open_price"),

    col("high_24h").cast("double").alias("high_price"),

    col("low_24h").cast("double").alias("low_price"),

    col("current_price").cast("double").alias("close_price"),

    col("total_volume").cast("double").alias("volume"),

    current_timestamp().alias("load_timestamp"),

    col("market_cap").cast("double").alias("market_cap")
)

In [0]:
incremental_silver_df.printSchema()
incremental_silver_df.show(5)
print("Rows:", incremental_df.count())

root
 |-- coin_symbol: string (nullable = true)
 |-- quote_currency: string (nullable = false)
 |-- observation_date: date (nullable = true)
 |-- open_price: double (nullable = true)
 |-- high_price: double (nullable = true)
 |-- low_price: double (nullable = true)
 |-- close_price: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)
 |-- market_cap: double (nullable = true)

+-----------+--------------+----------------+----------+----------+---------+-----------+---------------+--------------------+-----------------+
|coin_symbol|quote_currency|observation_date|open_price|high_price|low_price|close_price|         volume|      load_timestamp|       market_cap|
+-----------+--------------+----------------+----------+----------+---------+-----------+---------------+--------------------+-----------------+
|        BTC|           USD|      2026-10-10|      NULL|   83094.0|  82262.0|    82961.0|1.6109975008E10|2026-10-10 19:46:...

In [0]:
incremental_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("incremental_silver_market")

spark.sql("""
SELECT *
FROM incremental_silver_market
LIMIT 5
""").show()

+-----------+--------------+----------------+----------+----------+---------+-----------+---------------+--------------------+-----------------+
|coin_symbol|quote_currency|observation_date|open_price|high_price|low_price|close_price|         volume|      load_timestamp|       market_cap|
+-----------+--------------+----------------+----------+----------+---------+-----------+---------------+--------------------+-----------------+
|        BTC|           USD|      2026-10-10|      NULL|   83094.0|  82262.0|    82961.0|1.6109975008E10|2026-10-10 19:46:...|1.667205205117E12|
|        ETH|           USD|      2026-10-10|      NULL|   2513.03|  2474.44|    2510.77|  5.409131403E9|2026-10-10 19:46:...| 3.06620266049E11|
|       USDT|           USD|      2026-10-10|      NULL|  0.999347| 0.999124|    0.99917| 3.086203076E10|2026-10-10 19:46:...| 1.84099956114E11|
|        BNB|           USD|      2026-10-10|      NULL|     752.9|   737.32|     749.99|   4.94166138E8|2026-10-10 19:46:...|  9.

In [0]:
spark.sql("""

MERGE INTO silver_crypto_market AS target

USING incremental_silver_market AS source

ON target.coin_symbol = source.coin_symbol
AND target.observation_date = source.observation_date

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
SELECT *
FROM silver_crypto_market
WHERE observation_date = '2026-10-10'
LIMIT 20
""").show()

spark.sql("""
SELECT COUNT(*)
FROM silver_crypto_market
""").show()

+-----------+--------------+----------------+----------+----------+----------+-----------+---------------+--------------------+-----------------+
|coin_symbol|quote_currency|observation_date|open_price|high_price| low_price|close_price|         volume|      load_timestamp|       market_cap|
+-----------+--------------+----------------+----------+----------+----------+-----------+---------------+--------------------+-----------------+
|        BTC|           USD|      2026-10-10|      NULL|   83094.0|   82262.0|    82961.0|1.6109975008E10|2026-10-10 19:46:...|1.667205205117E12|
|        ETH|           USD|      2026-10-10|      NULL|   2513.03|   2474.44|    2510.77|  5.409131403E9|2026-10-10 19:46:...| 3.06620266049E11|
|       USDT|           USD|      2026-10-10|      NULL|  0.999347|  0.999124|    0.99917| 3.086203076E10|2026-10-10 19:46:...| 1.84099956114E11|
|        BNB|           USD|      2026-10-10|      NULL|     752.9|    737.32|     749.99|   4.94166138E8|2026-10-10 19:46:.

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp
import time

rows_processed = incremental_silver_df.count()
print(rows_processed)
run_id = int(time.time())
end_time = spark.sql(
    "SELECT current_timestamp()"
).first()[0]
log_data = [(
    run_id,
    "Incremental Merge",
    incremental_file,
    "SUCCESS",
    rows_processed,
    start_time,
    end_time
)]

log_df = spark.createDataFrame(
    log_data,
    [
        "run_id",
        "layer",
        "file_processed",
        "status",
        "rows_processed",
        "start_time",
        "end_time"
    ]
)

log_df = log_df.withColumn(
    "execution_time",
    current_timestamp()
)

250


In [0]:
log_df.write \
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable("pipeline_execution_logs")

In [0]:
spark.sql("""
SELECT *
FROM pipeline_execution_logs
""").show(truncate=False)

+----------+-----------------+---------------+-------+--------------+--------------------------+--------------------------+--------------------------+
|run_id    |layer            |file_processed |status |rows_processed|execution_time            |start_time                |end_time                  |
+----------+-----------------+---------------+-------+--------------+--------------------------+--------------------------+--------------------------+
|1791661601|Incremental Merge|markets(2).json|SUCCESS|250           |2026-10-10 19:47:24.99452 |2026-10-10 19:45:56.890369|2026-10-10 19:46:41.706223|
|1791661034|Incremental Merge|markets(2).json|SUCCESS|250           |2026-10-10 19:37:17.987634|NULL                      |NULL                      |
|1791657963|Incremental Merge|markets.json   |SUCCESS|250           |2026-10-10 18:46:04.808516|NULL                      |NULL                      |
|1791660990|Incremental Merge|markets.json   |SUCCESS|250           |2026-10-10 19:36:35.87819